# Semana 09: Infraestrutura como Código (IaC) com Terraform

## Módulo de Infraestrutura como Código — Fábrica Virtual Smart N1

Este notebook apresenta a **Infraestrutura como Código (IaC)**, o paradigma declarativo com a linguagem **HCL (HashiCorp Configuration Language)** no **Terraform**, o provisionamento de instâncias **AWS EC2** e Security Groups, e o gerenciamento de estado (`tfstate`).

### Objetivos de aprendizagem
- Compreender as vantagens da Infraestrutura como Código (IaC) em relação à configuração manual no console da AWS.
- Diferenciar a abordagem **Declarativa** (Terraform) da abordagem **Imperativa** (Scripts Bash/AWS CLI).
- Dominar os blocos fundamentais de código HCL (`provider`, `resource`, `variable`, `output`, `data`).
- Provisionar uma instância AWS EC2 com Docker pré-instalado via arquivo `user_data`.
- Entender os comandos do ciclo de vida Terraform: `init`, `plan`, `apply` e `destroy`.
- Simular o motor de reconciliação de estado em Python.

---


## 1. Fundamentação Teórica

### 1.1 O Paradigma da Infraestrutura como Código (IaC)

Com o Terraform, servidores, Security Groups, redes VPC e instâncias EC2 são declarados em **arquivos de texto versionados em Git**:

```text
  +-------------------------------------------------------------------------+
  |                    FLUXO DE TRABALHO DO TERRAFORM                       |
  |                                                                         |
  |  [Código main.tf (HCL)] ---> [terraform init]                           |
  |                                     |                                   |
  |                                     v                                   |
  |                             [terraform plan]                            |
  |                                     | (Gera Plano de Execução)          |
  |                                     v                                   |
  |                            [terraform apply]                            |
  |                                     |                                   |
  |            +------------------------+------------------------+          |
  |            |                                                 |          |
  |            v                                                 v          |
  |  [Atualiza terraform.tfstate]                     [Provisiona Instância |
  |  (Estado Atual da Infra)                           AWS EC2 com Docker]  |
  +-------------------------------------------------------------------------+
```

---

### 1.2 Exemplo de Arquivo HCL (`main.tf`) para AWS EC2

```hcl
# 1. Provider AWS
terraform {
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
  }
}

provider "aws" {
  region = "us-east-1"
}

# 2. Security Group (Liberando portas 22/SSH e 80/HTTP)
resource "aws_security_group" "sg_smartn1" {
  name        = "sg_smartn1_ec2"
  description = "Liberar tráfego HTTP e SSH para a fábrica"

  ingress {
    from_port   = 22
    to_port     = 22
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    from_port   = 80
    to_port     = 80
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  egress {
    from_port   = 0
    to_port     = 0
    protocol    = "-1"
    cidr_blocks = ["0.0.0.0/0"]
  }
}

# 3. Recurso: Instância AWS EC2 com Docker
resource "aws_instance" "servidor_devops" {
  ami           = "ami-0c7217cdde317cfec" # Ubuntu 22.04 LTS
  instance_type = "t3.micro"
  vpc_security_group_ids = [aws_security_group.sg_smartn1.id]

  user_data = <<-EOF
              #!/bin/bash
              apt-get update -y
              apt-get install -y docker.io
              systemctl start docker
              systemctl enable docker
              usermod -aG docker ubuntu
              EOF

  tags = {
    Name = "SmartN1_Servidor_Producao"
  }
}

output "ip_publico_ec2" {
  value = aws_instance.servidor_devops.public_ip
}
```

---


## 2. Prática — Simulação do Motor de Reconciliação do Terraform em Python

O motor do Terraform compara o **Estado Desejado** (código `.tf`) com o **Estado Atual** (`.tfstate`) para calcular o plano de alterações (`+ Criar`, `~ Modificar`, `- Destruir`). Vamos implementar essa lógica em Python.

In [ ]:
# Estado Desejado (definido no arquivo HCL .tf para AWS)
estado_desejado = {
    "aws_instance.servidor_devops": {"instance_type": "t3.micro", "ami": "ami-0c7217cdde317cfec", "status": "running"},
    "aws_security_group.sg_smartn1": {"ports": [22, 80], "status": "active"}
}

# Estado Atual (lido da infraestrutura AWS / terraform.tfstate)
estado_atual = {
    "aws_instance.servidor_devops": {"instance_type": "t2.micro", "ami": "ami-0c7217cdde317cfec", "status": "running"}, # Precisa modificar tipo
    "aws_security_group.sg_legacy": {"ports": [8080], "status": "active"}  # Deve ser destruído
}

def calcular_plano_terraform(desejado, atual):
    plano = {"criar": [], "modificar": [], "destruir": [], "sem_alteracao": []}
    
    for res_id, spec_desejada in desejado.items():
        if res_id not in atual:
            plano["criar"].append((res_id, spec_desejada))
        else:
            if spec_desejada == atual[res_id]:
                plano["sem_alteracao"].append(res_id)
            else:
                plano["modificar"].append((res_id, atual[res_id], spec_desejada))
                
    for res_id in atual:
        if res_id not in desejado:
            plano["destruir"].append(res_id)
            
    return plano

plano_execucao = calcular_plano_terraform(estado_desejado, estado_atual)

print("=== TERRAFORM PLAN SIMULADO (RECONCILIAÇÃO AWS) ===\n")
print(f"+ RECURSOS A CRIAR ({len(plano_execucao['criar'])}):", [r[0] for r in plano_execucao['criar']])
print(f"~ RECURSOS A MODIFICAR ({len(plano_execucao['modificar'])}):", [r[0] for r in plano_execucao['modificar']])
print(f"- RECURSOS A DESTRUIR ({len(plano_execucao['destruir'])}):", plano_execucao['destruir'])
print(f"= RECURSOS SEM ALTERAÇÃO ({len(plano_execucao['sem_alteracao'])}):", plano_execucao['sem_alteracao'])


---

## 3. Exercícios de Fixação e Avaliação

### Questão 1
Diferencie a abordagem **Declarativa** (utilizada pelo Terraform) da abordagem **Imperativa** (utilizada por scripts de automação em Bash ou AWS CLI). Por que a abordagem declarativa é mais segura para evitar inconsistências em infraestrutura na AWS?

### Questão 2
Para que serve o arquivo `terraform.tfstate`? Por que em equipes multidisciplinares esse arquivo deve ser mantido em um **Remote Backend** (ex: AWS S3 com trava DynamoDB) e nunca commitado diretamente no repositório Git?

### Questão 3
Explique o comando `terraform plan`. O que significa um plano indicar `Plan: 2 to add, 1 to change, 0 to destroy` ao provisionar recursos na AWS?
